# devbook_base2 — baseline notebook for the modular ld-decode

Three independent ways to get frames on the bench; run whichever cells fit the job:

- **Path A** — decode an `.lds`/`.ldf` inline with the modular decoder, writing **TBC** or **CVBS** output (your choice per cell).
- **Path B** — load frames from any existing `.tbc` **or** `.composite` via `analysis/tbc_frames.py` — both containers, NTSC and PAL, present the same interface.

Everything below Path B works on a `FrameSet` no matter how it was produced. The later sections are aimed at **comb filter experiments** (2D composite arrays + subcarrier/burst bookkeeping, with reference splits to compare against) and **VITS / test pattern measurements**.

In [ ]:
# Imports and paths

%load_ext autoreload
%autoreload 2

import os
import sys

import numpy as np
import scipy.signal as sps

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

try:
    # dark-mode plots to match dark notebooks
    from jupyterthemes import jtplot
    jtplot.style(theme='monokai', context='notebook', ticks=True, grid=False)
except Exception:
    pass

from PIL import Image
import IPython.display
from IPython.display import HTML

# repo paths: lddecode package, analysis/ toolkit
for p in ('..', '../lddecode', '../analysis'):
    module_path = os.path.abspath(p)
    if module_path not in sys.path:
        sys.path.insert(0, module_path)

# modular ld-decode (core.py is only a compat shim now)
from lddecode.decoder import LDdecode
from lddecode.fileio import make_loader
from lddecode.utils_logging import init_logging
from lddecode.utils_plotting import draw_field, plotline, RGBoutput

# frame access + VITS measurement toolkit (works on .tbc and .composite)
from tbc_frames import (
    load_frames,
    detect_patterns, summarize_patterns,
    measure_ntc7_multiburst, measure_ntc7_transients, measure_pal_its_transients,
    weighted_psnr, chroma_am_pm_noise, demod_region, burst_ref,
)

In [ ]:
# This needs to be done after the first block for some reason?
matplotlib.rc('figure', figsize=(40, 24))

In [ ]:
logger = init_logging(None)

## Path A — decode an `.lds`/`.ldf` inline

`rundecode` drives the modular `LDdecode` directly. `cvbs=False` writes `<outname>.tbc` + `.tbc.db`;
`cvbs=True` writes spec CVBS `<outname>.composite` + `.meta` instead.

Note for CVBS: the writer starts on a first field opening the colour sequence (phase ID 1), so it may
skip up to one frame at the start — decode one frame more than you need.

In [ ]:
def rundecode(filename, outname, firstframe, numframes, system='NTSC',
              cvbs=False, threads=4, extra_options=None):
    """Decode inline; returns (ldd, fields).

    cvbs=False -> <outname>.tbc/.tbc.db;  cvbs=True -> <outname>.composite/.meta.
    The returned fields are the in-memory Field objects (pre-output), useful
    with draw_field()/plotline().  ldd.close() finalizes the output files.
    """
    opts = {'threads': threads}
    if cvbs:
        opts['output_cvbs'] = True
    if extra_options:
        opts.update(extra_options)

    ldd = LDdecode(filename, outname, make_loader(filename), _logger=logger,
                   system=system, extra_options=opts)
    ldd.roughseek(firstframe * 2)
    if system == 'NTSC':
        ldd.blackIRE = 7.5

    fields = []
    while len(fields) < numframes * 2:
        f = ldd.readfield()
        if f is None:
            break
        fields.append(f)

    ldd.close()   # writes .tbc.db / closes .composite+.meta
    return ldd, fields

In [ ]:
# Pick a source and system (repo testdata shown; point at your own captures as needed)
filename, system = '../testdata/pal/jason-testpattern.ldf', 'PAL'
# filename, system = '../testdata/ntsc/ve-snw-cut.ldf', 'NTSC'

outname = 'devbook'

# A1: decode to TBC
ldd, decoded_fields = rundecode(filename, outname, 0, 3, system)

In [ ]:
# A2 (optional, at your discretion): same decode but writing CVBS output
ldd_c, decoded_fields_c = rundecode(filename, outname + '_cvbs', 0, 4, system, cvbs=True)

In [ ]:
# Displays a decoded field straight from the decoder (Path A only);
# plotline(decoded_fields[0], 19) plots the pre-TBC demod of one line.
draw_field(decoded_fields[0])

In [ ]:
# A3: get a FrameSet directly from the in-memory decoded fields — no file
# round-trip, and independent of which output mode (if any) was written.
# It is the same object Path B produces, so every section below works on it
# unchanged; each field also keeps the full decoder Field as .decoded
# (linelocs, demod data, plotline(...), etc.).
from tbc_frames import frames_from_decoded

frames = frames_from_decoded(decoded_fields)
frames

## Path B — load frames with `tbc_frames`

Works identically on `.tbc` and `.composite`, PAL and NTSC — from the decode above or any existing file.
A `FrameSet` is a sequence of `Frame`s (paired fields); each field is a `FieldView` with 2D arrays and
chroma bookkeeping, and passes directly into the tbc_common/CombNTSC measurement machinery.

In [ ]:
# Point at whichever output you want on the bench:
frames = load_frames(outname + '.tbc', n_frames=3)
# frames = load_frames(outname + '_cvbs.composite', n_frames=3)
# frames = load_frames('../cbar_he.tbc', n_frames=4)          # existing NTSC bars decode
# frames = load_frames('../jasonbars.composite', n_frames=3)  # existing PAL CVBS

frames

In [ ]:
fr = frames[0]

plt.figure(figsize=(16, 9))
plt.imshow(fr.interlaced(), cmap='gray', aspect='auto', interpolation='nearest')
plt.title(f'{frames.system} frame {fr.index} (plain weave, IRE)')
plt.colorbar(shrink=0.6)
plt.show()

## Comb filter experiments

`FieldView` geometry cheat-sheet (both containers, 4×fsc sampling):

- subcarrier at exactly **fs/4** — `fr.first.carrier()` gives the quadrature reference with the same
  sign convention as `demod_region`;
- absolute phase is per-line only — reference to the same line's burst (`burst_phasor[s]`);
- stored-line-to-line phase advance within a field: **NTSC 180°/line**, **PAL 270°/line + V-switch**;
  PAL CVBS line starts additionally drift +0.0064 samples/line (90° of subcarrier per sample);
- `split_comb1d()` is the exact-at-fs/4 horizontal baseline — the reference to beat.

In [ ]:
# Baseline: 1D horizontal split
y, c = fr.first.split_comb1d()

fig, ax = plt.subplots(1, 2, figsize=(20, 7))
ax[0].imshow(y, cmap='gray', aspect='auto', interpolation='nearest')
ax[0].set_title('luma (1D comb baseline)')
ax[1].imshow(np.abs(c), cmap='gray', aspect='auto', interpolation='nearest', vmax=25)
ax[1].set_title('|chroma| (1D comb baseline)')
plt.show()

In [ ]:
# Scaffold for your own comb — work on the 2D composite and compare to the baseline.
# Example: simple intra-field vertical (line) comb.  NTSC stored lines alternate
# 180 deg, so neighbour averaging cancels chroma; on PAL (270 deg/line) a 2-line
# delay comb needs the V-switch handled — this cell is the starting point, not the answer.
x = fr.first.ire

if frames.system == 'NTSC':
    cv = np.zeros_like(x)
    cv[1:-1] = (x[1:-1] - (x[:-2] + x[2:]) / 2) / 2
    yv = x - cv

    line = 120
    plt.figure(figsize=(16, 5))
    plt.plot(y[line], label='1D comb luma')
    plt.plot(yv[line], label='vertical comb luma')
    plt.plot(x[line], alpha=0.3, label='composite')
    plt.legend(); plt.title(f'field line {line + 1}')
    plt.show()
else:
    # PAL: start burst-relative.  Per-line burst phasors fold out the lattice/line
    # phase so U/V demod can be compared line to line.
    bp = fr.first.burst_phasors()
    print('burst amp IRE: median %.1f' % np.median(np.abs(bp[np.abs(bp) > 0])))

In [ ]:
# NTSC reference comb: lddecode's CombNTSC (1D, and 3D with the previous frame)
if frames.system == 'NTSC' and len(frames) >= 2:
    comb1 = frames[1].comb_ntsc()
    comb3 = frames[1].comb_ntsc(prev_frame=frames[0])
    print('line 19 (level, phase, SNR)  1D:', comb1.calcLine19Info())
    print('line 19 (level, phase, SNR)  3D:', comb3.calcLine19Info())

In [ ]:
# Per-line colour burst: amplitude and phase down the field
bp = fr.second.burst_phasors()
lines = np.arange(1, len(bp) + 1)
ok = np.abs(bp) > 2

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(lines[ok], np.abs(bp)[ok], '.')
ax[0].set_title('burst amplitude (IRE) by field line')
ax[1].plot(lines[ok], np.degrees(np.angle(bp))[ok], '.')
ax[1].set_title('burst phase (deg) by field line')
plt.show()

## VITS / test pattern measurements

`frames.detect()` reports what's on the disc; the measurement helpers below take any field
(`FieldView` works directly). Add further processing here — everything from `tbc_common`
(NTC-7, ITS, weighted PSNR, chroma AM/PM noise, transients, colour bars via
`analysis/smpte_analyze.py`) is importable from `tbc_frames`.

In [ ]:
det = frames.detect()

In [ ]:
# Frequency response from multiburst (NTSC NTC-7 rides second-field line 20);
# PAL test discs: ITS 2T/bar transients instead.
if frames.system == 'NTSC':
    packets = measure_ntc7_multiburst(fr.second)
    if packets:
        f = [p[1] for p in packets]
        a = [p[2] for p in packets]
        plt.figure(figsize=(8, 5))
        plt.plot(f, 20 * np.log10(np.array(a) / a[0]), 'o-')
        plt.xlabel('MHz'); plt.ylabel('dB vs first packet')
        plt.title('NTC-7 multiburst response'); plt.grid(True)
        plt.show()
    else:
        print('no multiburst found on this field')
else:
    seconds = [fv.field for fv in frames.fields if not fv.isFirstField]
    print('ITS transients:', measure_pal_its_transients(seconds))

In [ ]:
# Weighted SNR of a flat region (pick a line/window with flat luma on your disc)
firsts = [fv.field for fv in frames.fields if fv.isFirstField]
print('weighted / unweighted PSNR, mean IRE:',
      weighted_psnr(firsts, line=13, start_us=20, duration_us=20))

In [ ]:
# Full chroma decode + display via ld-chroma-decoder (TBC output only, binary must be in PATH)
rgb = RGBoutput(outname)
rgb.display(0)